# Task 15: 14-species CV, Final-B and grayscale/16px runs (07-11-2026)

Supervisor requests 6–7 (06-11-2026), on the 14 clean species (17 minus *Callitris columellaris*,
*Cunninghamia lanceolata*, *Fokienia hodginsii*; 4,115 images, 140 origins):

| Request | Runs | Output (`data/experiments/`) |
|---|---:|---|
| 6, CV | 5 models × 5 folds = 25 | `Task-B-clean14-cv/<model>/fold_<k>/` |
| 6, Final-B | 5 models × seeds 42/123/456 = 15 | `Task-B-clean14-final/seed_<s>/<model>/` |
| 7 | ConvNeXt × 5 folds × {grayscale, px16} = 10 | `Task-B-clean14-cv-{grayscale,px16}/` |

Rule for all 50 runs: backbone and head LR 1e-4, 5 epochs linear warmup + cosine, at most 50 epochs,
`best.pth` = lowest val loss from epoch 6 (warmup excluded), early stopping patience 15 on val loss,
held-out fold (CV) or val + test (Final-B) scored once with `best.pth`. CV folds are the CV v2 folds
drawn on 17 species, then the 3 species are dropped (`origin_cv.drop_species`).

This notebook is the record of the run. It starts `run_clean14.sh` (22 jobs, `MAX_PARALLEL` at a time),
prints the launcher events, the latest epoch of every running job, each finished run's result and the
files it wrote, every `POLL_SECONDS`, then builds the tables with `summarize_clean14.py` and copies the
logs into `artifacts/`. Full per-job logs: `data/experiments/clean14-logs/<UTC>/<job>.log`.
Runbook: `docs/timeline/07-11-2026-handoff-vast-clean14.md`.

Run headless (papermill saves the executed copy while a cell runs):

```bash
cd "notebooks/phase 2/supervisor-feedback/task_15_clean14_runs"
papermill clean14_runs.ipynb \
  ../../../../artifacts/phase_2/supervisor-feedback/task_15_clean14_runs/clean14_runs_executed.ipynb \
  -k python3 --cwd .
```

Re-running is safe: finished folds/runs (with `metrics.json`) are kept; unfinished ones restart at epoch 1.

In [ ]:
from pathlib import Path
import sys

# SSH server: detect from notebook location (cwd)
ROOT = Path.cwd().parents[3]  # notebooks/phase 2/supervisor-feedback/task_15_clean14_runs/ -> woodbench/
assert (ROOT / "src").is_dir() and (ROOT / "configs").is_dir(), ROOT

sys.path.insert(0, str(ROOT))

import torch
print(f"[ROOT] {ROOT}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Parameters (papermill can override)
MAX_PARALLEL = 5
POLL_SECONDS = 300
PHASES = ""  # "" = cv degraded final; or e.g. "final"

In [ ]:
import json
import os
import platform
import re
import shutil
import subprocess
import time
from datetime import datetime, timezone

import pandas as pd
import timm
import torchvision
from IPython.display import Markdown, display

assert torch.cuda.is_available(), "CUDA is required"
TASK_DIR = Path.cwd()
PYTHON = sys.executable
DATA_ROOT = ROOT / "data/task-B-group-split-224"
EXP_ROOT = ROOT / "data/experiments"
ARTIFACTS = ROOT / "artifacts/phase_2/supervisor-feedback/task_15_clean14_runs"
CV_ROOTS = {"rgb": "Task-B-clean14-cv", "grayscale": "Task-B-clean14-cv-grayscale", "px16": "Task-B-clean14-cv-px16"}
FINAL_ROOT = EXP_ROOT / "Task-B-clean14-final"


def sh(*command, cwd=ROOT):
    return subprocess.run(command, cwd=cwd, capture_output=True, text=True).stdout.strip()


def utc_now():
    return datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")


print("started:", utc_now())
print("commit:", sh("git", "rev-parse", "HEAD"), "| branch:", sh("git", "branch", "--show-current"))
print("git status (tracked):", sh("git", "status", "--short", "--untracked-files=no") or "clean")
print(f"python {platform.python_version()} ({PYTHON})")
print(f"torch {torch.__version__} (CUDA {torch.version.cuda}), torchvision {torchvision.__version__}, timm {timm.__version__}")
print("Note: the handoff names torch 2.15.0+cu132; this host driver supports CUDA 12.7 only, so the")
print("      cu126 build of torch 2.14.1 is used (handoff section 3.2 allows this).")
print(sh("nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader"))
print(f"CPU cores: {os.cpu_count()}")
for split in ("train", "val", "test"):
    print(f"{split}: {sum(1 for _ in (DATA_ROOT / split).rglob('*.png'))} images (cache, 17 species)")

## 1. Start the launcher

`run_clean14.sh` checks the 224 cache against `docs/timeline/01-11-2026-task-B-group-split-224.sha256`,
records commit, `pip freeze` and `nvidia-smi` in its log folder, then queues the jobs. It runs in its own
session, so training continues even if this kernel stops; re-run this notebook to resume monitoring.

In [ ]:
class Attached:
    # A launcher started by an earlier run of this notebook (watched by pid).
    def __init__(self, pid):
        self.pid, self.returncode = pid, None

    def poll(self):
        if Path(f"/proc/{self.pid}").exists():
            return None
        status = launch_out.read_text()
        self.returncode = 0 if re.search(r"^All jobs done; 0 failed", status, re.M) else 1
        return self.returncode


running_pids = sh("pgrep", "-f", r"^bash .*/run_clean14\.sh").split()
if running_pids:
    # Never start a second launcher: it would delete runs the first one is training.
    launch_out = sorted((EXP_ROOT / "clean14-logs").glob("launch-*.out"))[-1]
    launcher = Attached(int(running_pids[0]))
    print(f"ATTACHED to running launcher pid {launcher.pid}; stdout -> {launch_out.relative_to(ROOT)}")
else:
    launch_out = EXP_ROOT / "clean14-logs" / f"launch-{datetime.now(timezone.utc):%Y%m%dT%H%M%SZ}.out"
    launch_out.parent.mkdir(parents=True, exist_ok=True)
    env = {**os.environ, "PYTHON": PYTHON, "MAX_PARALLEL": str(MAX_PARALLEL), "PYTHONUNBUFFERED": "1"}
    launcher = subprocess.Popen(
        ["bash", str(TASK_DIR / "run_clean14.sh"), *PHASES.split()],
        cwd=TASK_DIR, env=env, stdout=open(launch_out, "w"), stderr=subprocess.STDOUT,
        stdin=subprocess.DEVNULL, start_new_session=True,
    )
    print(f"launcher pid {launcher.pid}; stdout -> {launch_out.relative_to(ROOT)}")

LOG_DIR = None
for _ in range(600):
    text = launch_out.read_text()
    match = re.search(r"^Logs: (.+)$", text, re.M)
    if match and "jobs, at most" in text:
        LOG_DIR = Path(match.group(1))
        break
    if launcher.poll() is not None:
        break
    time.sleep(1)
print(launch_out.read_text())
assert LOG_DIR is not None, "launcher stopped before queuing jobs"
JOBS = (LOG_DIR / "jobs.txt").read_text().split()
print(f"log folder: {LOG_DIR.relative_to(ROOT)}")
print((LOG_DIR / "launch.txt").read_text())

## 2. Monitor

Every `POLL_SECONDS`: launcher events, GPU load, latest epoch of each running job (`[best]` = new lowest
val loss, `(warmup)` = epochs 1–5, not eligible), and every run that finished since the last poll with its
result and the files it wrote. Val numbers during training are AMP batch means; the reported ones are in
`metrics.json`.

In [ ]:
def run_label(path):
    rel = path.parent.relative_to(EXP_ROOT).parts
    if rel[0] == FINAL_ROOT.name:
        return f"final {rel[1]} {rel[2]}"
    variant = next(k for k, v in CV_ROOTS.items() if v == rel[0])
    return f"cv {variant} {rel[1]} {rel[2]}"


def run_result(path):
    m = json.loads(path.read_text())
    if "test" in m:
        return (f"best epoch {m['best_epoch']}/{m['epochs_trained']}: val acc {m['val']['accuracy'] * 100:.2f}%, "
                f"test acc {m['test']['accuracy'] * 100:.2f}%, test macro-F1 {m['test']['f1_macro']:.4f}")
    return (f"best epoch {m['best_epoch']}/{m['epochs_trained']}: held-out acc {m['accuracy'] * 100:.2f}%, "
            f"macro-F1 {m['f1_macro']:.4f}")


def run_files(folder):
    items = []
    for f in sorted(p for p in folder.iterdir() if p.is_file()):
        size = f.stat().st_size
        items.append(f"{f.name} ({size / 1e6:.0f} MB)" if size > 1e6 else f.name)
    return ", ".join(items)


def finished_runs():
    paths = [p for root in CV_ROOTS.values() for p in (EXP_ROOT / root).glob("*/fold_*/metrics.json")]
    paths += list(FINAL_ROOT.glob("seed_*/*/metrics.json"))
    return set(paths)


def last_epoch_line(job):
    log = LOG_DIR / f"{job}.log"
    if not log.exists():
        return "(no log yet)"
    lines = [l.strip() for l in re.split(r"[\r\n]", log.read_text(errors="replace")) if l.strip()]
    epoch = [l for l in lines if re.search(r"epoch \d+/\d+", l)]
    return epoch[-1] if epoch else (lines[-1][:140] if lines else "(empty log)")


seen_runs = set()
seen_events = 0
start = time.time()
while True:
    done = launcher.poll() is not None
    print(f"===== {utc_now()} | elapsed {(time.time() - start) / 60:.0f} min =====")
    events = launch_out.read_text().splitlines()
    events = [e for e in events if re.match(r"\d\d:\d\d:\d\d ", e) or e.startswith("All jobs")]
    for event in events[seen_events:]:
        print("  launcher:", event)
    seen_events = len(events)
    exits = {p.stem: p.read_text().strip() for p in LOG_DIR.glob("*.exit")}
    started = {e.split()[1] for e in events if e.endswith(" started")}
    running = [j for j in JOBS if j in started and j not in exits]
    failed = sorted(j for j, s in exits.items() if s != "0")
    gpu = sh("nvidia-smi", "--query-gpu=utilization.gpu,memory.used,memory.total", "--format=csv,noheader")
    print(f"  jobs: {len(exits)}/{len(JOBS)} done, {len(running)} running, {len(failed)} failed {failed or ''}"
          f" | runs with metrics.json: {len(finished_runs())}/50 | GPU {gpu}")
    for job in running:
        print(f"  {job:>28}: {last_epoch_line(job)}")
    for path in sorted(finished_runs() - seen_runs):
        print(f"  NEW {run_label(path)}: {run_result(path)}")
        print(f"      {path.parent.relative_to(ROOT)}/: {run_files(path.parent)}")
    seen_runs = finished_runs()
    if done:
        break
    time.sleep(POLL_SECONDS)
print(f"launcher exit code {launcher.returncode}")

In [ ]:
exits = {p.stem: p.read_text().strip() for p in LOG_DIR.glob("*.exit")}
status = pd.DataFrame({"job": JOBS, "exit": [exits.get(j, "missing") for j in JOBS]})
print(status.to_string(index=False))
bad = status[status.exit != "0"]
for job in bad.job:
    print(f"\n--- tail of {job}.log ---")
    print("\n".join((LOG_DIR / f"{job}.log").read_text(errors="replace").splitlines()[-30:]))
assert bad.empty, f"{len(bad)} job(s) failed; fix and re-run this notebook (finished runs are kept)"
assert len(finished_runs()) == 50, len(finished_runs())
print("All 22 jobs exited 0; 50 runs have metrics.json")

## 3. Tables

`summarize_clean14.py` exports the CV reports (registry, OOF, per-species, as for CV v2) and writes
`artifacts/phase_2/supervisor-feedback/task_15_clean14_runs/results/`. SD is the sample SD over folds
or seeds; it describes variation, not a confidence interval. One Final-B test image = 0.23 points (432
images); differences of 2–3 images are noise.

In [ ]:
result = subprocess.run([PYTHON, "summarize_clean14.py"], cwd=TASK_DIR, capture_output=True, text=True)
print(result.stdout[-6000:])
print(result.stderr[-3000:])
assert result.returncode == 0, "summarize_clean14.py failed"
RESULTS = ARTIFACTS / "results"
display(Markdown((RESULTS / "summary.md").read_text()))

In [ ]:
pd.set_option("display.width", 200, "display.max_columns", 30)
for name in ("cv_summary", "cv_vs_17species", "degraded_summary", "final_b_summary", "final_b_runs"):
    print(f"\n### {name}.csv")
    display(pd.read_csv(RESULTS / f"{name}.csv"))

## 4. Keep the logs with the artifacts

Copies `data/experiments/clean14-logs/` (launcher output, per-job logs and exit codes, `pip_freeze.txt`,
`nvidia-smi.txt`) to `artifacts/.../task_15_clean14_runs/logs/` and lists what goes into git.
Checkpoints (`best.pth`) stay out of git; `results/checkpoints.sha256` lists them and they are backed up
to Drive (`drive:wood/2026-11-07-clean14-runs/`, handoff section 6.3).

In [ ]:
shutil.copytree(EXP_ROOT / "clean14-logs", ARTIFACTS / "logs", dirs_exist_ok=True)
rows = []
for f in sorted(p for p in ARTIFACTS.rglob("*") if p.is_file()):
    rows.append({"file": str(f.relative_to(ARTIFACTS)), "kB": round(f.stat().st_size / 1e3, 1)})
listing = pd.DataFrame(rows)
print(f"{len(listing)} files, {listing.kB.sum() / 1e3:.1f} MB under {ARTIFACTS.relative_to(ROOT)}")
top = listing.assign(group=listing.file.str.split("/").str[:2].str.join("/")).groupby("group").agg(files=("file", "size"), kB=("kB", "sum"))
display(top)
n_ckpt = len((RESULTS / "checkpoints.sha256").read_text().splitlines())
ckpt_gb = sum(p.stat().st_size for p in EXP_ROOT.glob("Task-B-clean14-*/**/best.pth")) / 1e9
print(f"checkpoints.sha256: {n_ckpt} entries; best.pth on disk: {ckpt_gb:.1f} GB")
print("finished:", utc_now())